# Prompt Engineering Guide

If LLMs are engines, then prompts are the steering wheel. **Prompt engineering** is the art (and increasingly, science) of crafting inputs to get the most accurate, useful, and consistent outputs from LLMs.

Why does this matter? Because the *same* model can give you a mediocre answer or a brilliant one — the difference is entirely in how you ask. A well-crafted prompt can turn a $0 local model into something that rivals expensive APIs.

In this notebook, we'll walk through the major prompting techniques, see them in action, and understand **why** each one works at a deeper level.

In [1]:
import ollama
MODEL = 'llama3'  # Change this if you pulled a different model

## 1. Zero-Shot Prompting

The simplest technique: just ask the model directly, with **no examples**. The model relies entirely on its pre-training knowledge.

### Why it works
During pre-training, the model saw billions of instruction-response pairs. It learned general patterns like "when someone says 'translate X to Y', they want a translation." Zero-shot works well for tasks the model has seen tons of examples of during training.

### When it fails
Zero-shot struggles with:
- Unusual output formats the model hasn't seen before
- Domain-specific tasks (legal, medical, geospatial jargon)
- Tasks requiring very specific logic or reasoning

In [2]:
# Simple zero-shot: the model has seen millions of translation examples
prompt = "Translate 'Hello, how are you?' to Spanish."

response = ollama.generate(model=MODEL, prompt=prompt)
print(f"Prompt: {prompt}")
print(f"Response: {response['response']}")

Prompt: Translate 'Hello, how are you?' to Spanish.
Response: A classic!

The translation of "Hello, how are you?" to Spanish is:

"Hola, ¿cómo estás?"

Here's a breakdown:

* "Hola" means "hello".
* "¿Cómo estás?" is a polite way to ask "how are you?" (literally "how are you?").

So, when you put them together, you get the friendly greeting: "Hola, ¿cómo estás?"


In [3]:
# Zero-shot for classification — works but output format may vary
prompt = "Classify this satellite image description as either 'urban', 'agricultural', or 'forest': 'Dense tree canopy with no visible roads or structures.'"

response = ollama.generate(model=MODEL, prompt=prompt)
print(f"Response: {response['response']}")

Response: I would classify this satellite image description as "forest". The presence of a dense tree canopy and the absence of roads or structures suggest a natural environment with minimal human impact, characteristic of a forest.


**Notice** how the model might give you a paragraph instead of just the label? That's a classic zero-shot issue. Few-shot prompting fixes this.

## 2. Few-Shot Prompting

Provide a few **examples** of input → output pairs before your actual question. This teaches the model the exact pattern you want.

### Why it works
LLMs are incredible pattern matchers. When they see 2-3 examples, they pick up on:
- The **format** you want (just a label? a full sentence?)
- The **logic** of the task (what constitutes positive vs. negative?)
- The **style** of the response (formal? casual? terse?)

This is called **in-context learning** — the model "learns" temporarily from the examples, without any weight updates.

In [4]:
# Without examples — the model might be verbose
zero_shot = "Classify this review: 'It's okay, could be better.'"
response = ollama.generate(model=MODEL, prompt=zero_shot)
print("ZERO-SHOT result:")
print(response['response'])
print()

ZERO-SHOT result:
A classic!

I would classify this review as:

**Neutral**

The reviewer doesn't seem to have strong feelings about the product or service one way or another. The phrase "could be better" suggests that they had some issues or expectations that weren't met, but overall it's a pretty lukewarm assessment.



In [5]:
# With 2 examples — the model follows the exact pattern
few_shot = """Classify the sentiment of each review.

Input: Great product, works perfectly! -> Sentiment: Positive
Input: Terrible, it broke immediately. -> Sentiment: Negative
Input: It's okay, could be better. -> Sentiment: """

response = ollama.generate(model=MODEL, prompt=few_shot)
print("FEW-SHOT result:")
print(response['response'])

FEW-SHOT result:
Neutral


In [6]:
# Few-shot for a geospatial task: land cover classification from descriptions
geo_prompt = """Classify the land cover type from the description.

Description: Rows of crops visible, center-pivot irrigation pattern. -> Class: Agricultural
Description: Dense high-rise buildings, complex road network. -> Class: Urban
Description: Mixed broadleaf canopy with river running through. -> Class: Forest
Description: Large flat reflective surface, surrounded by barren land. -> Class: """

response = ollama.generate(model=MODEL, prompt=geo_prompt)
print(f"Classification: {response['response']}")

Classification: Based on the description, I would classify it as:

-> Class: Water (e.g., lake, reservoir, or pond)


**Pro tip:** 2-5 examples is usually enough. More examples give diminishing returns and eat into your context window.

## 3. Chain-of-Thought (CoT) Prompting

This is one of the most important discoveries in prompt engineering. Instead of asking for just the answer, you ask the model to **think step by step**.

### Why it works
LLMs generate tokens left-to-right. When the model writes out intermediate steps, those steps become part of its "memory" for the next token. It's essentially giving itself scratchpad space.

Without CoT: The model tries to jump directly to the answer and often gets complex problems wrong.
With CoT: The model breaks the problem down, and each step informs the next.

### The magic words that changed AI
Google researchers found that adding just **"Let's think step by step."** to a prompt dramatically improves accuracy on math and logic tasks. This simple phrase is one of the most cited discoveries in AI research.

In [7]:
# WITHOUT Chain-of-Thought
direct_prompt = """If a satellite captures one image every 5 days, and we need 
at least 12 cloud-free images for analysis, and on average 40% of images are cloudy, 
how many days of data collection do we need?"""

response = ollama.generate(model=MODEL, prompt=direct_prompt, options={'temperature': 0.1})
print("WITHOUT Chain-of-Thought:")
print(response['response'])

WITHOUT Chain-of-Thought:
Let's break this down step by step:

1. The satellite captures one image every 5 days.
2. We need at least 12 cloud-free images for analysis.
3. On average, 40% of the images are cloudy.

To find out how many images we need to collect in total, let's calculate the number of cloudy images:

Number of cloudy images = Total images taken × Percentage of cloudy images
= (Total images taken) × 0.4

Since we don't know the total number of images yet, let's call it x. Then:

x × 0.4 = Number of cloudy images

We need at least 12 cloud-free images, so the number of non-cloudy images must be at least 12. Let's call this number y.

y = Total images taken - Number of cloudy images
= x - (x × 0.4)

Since we need at least 12 cloud-free images, we set y ≥ 12:

x - (x × 0.4) ≥ 12

Simplifying the inequality, we get:

0.6x ≥ 12

Dividing both sides by 0.6, we find:

x ≥ 20

So, we need to collect at least 21 images in total.

Now, let's calculate how many days of data collecti

In [8]:
# WITH Chain-of-Thought — same problem, much better reasoning
cot_prompt = """If a satellite captures one image every 5 days, and we need 
at least 12 cloud-free images for analysis, and on average 40% of images are cloudy, 
how many days of data collection do we need?

Let's think step by step."""

response = ollama.generate(model=MODEL, prompt=cot_prompt, options={'temperature': 0.1})
print("WITH Chain-of-Thought:")
print(response['response'])

WITH Chain-of-Thought:
Let's break it down step by step.

1. The satellite captures one image every 5 days.
2. We need at least 12 cloud-free images for analysis.
3. On average, 40% of the images are cloudy. This means that 60% of the images (100% - 40%) are cloud-free.

Let's calculate how many images we need to capture in total:

Since we need 12 cloud-free images, and each image is captured every 5 days, we can divide 12 by 0.6 (the proportion of cloud-free images) to get the total number of images needed:

12 / 0.6 = 20

So, we need a minimum of 20 images in total.

Now, let's calculate how many days it takes to capture these 20 images:

Since each image is captured every 5 days, we can multiply the total number of images by 5:

20 x 5 = 100

Therefore, we need at least 100 days of data collection to capture a minimum of 12 cloud-free images.


## 4. System Prompts: The Persona

A **system prompt** is a special message that sets the model's behavior, personality, and constraints for the entire conversation. It's like giving the model a job description before it starts working.

### Why it works
During instruction tuning, models were trained to follow system prompts faithfully. The system message sits at the top of the context and influences every response. It's much more effective than trying to embed instructions in each user message.

### What makes a great system prompt?
1. **Role**: Who is the model? ("You are a senior GIS analyst...")
2. **Constraints**: What should it NOT do? ("Do not make up data...")
3. **Format**: How should it respond? ("Always respond in bullet points...")
4. **Context**: What background info does it need?

In [9]:
# Fun example: the pirate persona
response = ollama.chat(model=MODEL, messages=[
    {'role': 'system', 'content': 'You are a helpful pirate. Answer all questions with pirate slang and nautical metaphors.'},
    {'role': 'user', 'content': 'How do LLMs work?'}
])

print("🏴‍☠️ Pirate AI:")
print(response['message']['content'])

🏴‍☠️ Pirate AI:
Arrr, ye be wantin' to know the secrets o' Large Language Models, eh? Alright then, matey! I'll give ye the lowdown.

LLMs be like mighty ships on the seas o' knowledge, navigatin' through treacherous waters o' text data. They use a combination o' clever algorithms and cleverer folk (that be us programmers) to learn from vast libraries o' texts, just like how ye learn the ways o' the sea by readin' charts and listenin' to old salts.

These language models be trained on massive collections o' words, phrases, and sentences, like a treasure chest overflowin' with golden doubloons. They use this booty to teach themselves patterns, relationships, and hidden meanings between words, just as ye learn the tides and the winds by watchin' the stars and feelin' the sea breeze.

As they learn, LLMs improve their ability to predict what comes next in a sentence or text, like anticipatin' the weather by readin' the signs o' nature. This be known as "predictive modeling," me hearty! It

In [10]:
# Practical example: a geospatial analyst persona
response = ollama.chat(model=MODEL, messages=[
    {'role': 'system', 'content': """You are a Senior Geospatial Data Analyst with 15 years of experience.
Your expertise includes remote sensing, GIS, satellite imagery analysis, and spatial statistics.
When answering:
- Use precise technical terminology
- Reference specific tools (QGIS, GDAL, Rasterio) when relevant
- Give practical, actionable advice
- Mention limitations or caveats when they exist"""},
    {'role': 'user', 'content': 'What is GeoJSON and when should I use it?'}
])

print("🌍 Geo Analyst AI:")
print(response['message']['content'])

🌍 Geo Analyst AI:
GeoJSON! It's a fantastic format for exchanging geospatial data between different systems, services, and languages. As a Senior Geospatial Data Analyst, I'm delighted to share my insights on what GeoJSON is and when you should use it.

What is GeoJSON?
-------------------

GeoJSON (Geographic JSON) is an open standard for representing simple geographical features, such as points, lines, polygons, and multi-part geometries, using the JavaScript Object Notation (JSON). It's a lightweight, human-readable format that can be easily parsed by various programming languages.

Key Features:

* Supports GeoJSON coordinates in both WGS84 (EPSG:4326) and projected coordinate systems
* Can represent simple features like points, lines, polygons, and multi-part geometries
* Includes support for metadata, such as timestamps, IDs, and attributes

When to Use GeoJSON?
--------------------

GeoJSON is an excellent choice when:

1. **Interoperability**: You need to exchange geospatial da

## 5. Prompt Structuring: Best Practices

Beyond the major techniques, there are structural tricks that make prompts much more effective:

### Use Delimiters
Separate different parts of your prompt with clear markers. This prevents the model from confusing instructions with data.

In [11]:
# Bad: instructions and data mixed together
bad_prompt = "Summarize this text which is about climate change impacts on coastal cities and rising sea levels"

# Good: clear separation with delimiters
good_prompt = """Summarize the following text in 2 sentences.

---
TEXT:
Climate change is causing significant impacts on coastal cities worldwide. 
Rising sea levels threaten infrastructure, while increased storm intensity 
compounds flooding risks. In Karachi alone, projections suggest a 0.5m 
sea level rise by 2050 could affect 2.5 million people.
---

Summary:"""

response = ollama.generate(model=MODEL, prompt=good_prompt)
print(response['response'])

Here is a summary of the text in 2 sentences:

Climate change is having significant effects on coastal cities globally, including rising sea levels and increased storm intensity that pose threats to infrastructure and flooding risks. In Karachi, for example, a projected 0.5m sea level rise by 2050 could impact approximately 2.5 million people.


### Specify Output Format

Be explicit about exactly what format you want. Don't leave it to chance.

In [12]:
# Vague prompt — unpredictable format
vague = "Tell me about spectral indices used in remote sensing."

# Specific prompt — controlled format
specific = """List 3 spectral indices used in remote sensing.

For each index, provide:
- Name
- Formula (using band names like RED, NIR, SWIR)
- Primary use case (1 sentence)

Format as a numbered list."""

response = ollama.generate(model=MODEL, prompt=specific, options={'temperature': 0.2})
print(response['response'])

Here are three spectral indices used in remote sensing:

1. **Normalized Difference Vegetation Index (NDVI)**
Formula: (NIR - RED) / (NIR + RED)
Primary use case: NDVI is commonly used to detect and monitor vegetation health, density, and biomass.

2. **Spectral Index of Soil Moisture (SIM)**
Formula: (SWIR1 - SWIR2) / (SWIR1 + SWIR2)
Primary use case: SIM is used to estimate soil moisture levels, which is important for agricultural management, hydrological modeling, and climate studies.

3. **Burned Area Index (BAI)**
Formula: (NIR - RED) / RED
Primary use case: BAI is used to detect and map burned areas, helping firefighters and land managers track the extent of wildfires and monitor post-fire recovery.

Note: Band names like RED, NIR, and SWIR refer to specific wavelength ranges in the electromagnetic spectrum.


## 6. Comparing Techniques on the Same Task

Let's see all our techniques side-by-side on one task to really understand the differences.

In [13]:
task = "What is the capital of Australia?"

# 1. Zero-shot
r1 = ollama.generate(model=MODEL, prompt=task, options={'temperature': 0.1})
print("1. ZERO-SHOT:")
print(f"   {r1['response'][:200]}")
print()

# 2. Few-shot
few_shot_task = """Answer the geography question with ONLY the city name.

Q: What is the capital of France? A: Paris
Q: What is the capital of Japan? A: Tokyo
Q: What is the capital of Australia? A: """
r2 = ollama.generate(model=MODEL, prompt=few_shot_task, options={'temperature': 0.1})
print("2. FEW-SHOT:")
print(f"   {r2['response'][:200]}")
print()

# 3. CoT
cot_task = """What is the capital of Australia? 
Many people get this wrong. Let's think step by step."""
r3 = ollama.generate(model=MODEL, prompt=cot_task, options={'temperature': 0.1})
print("3. CHAIN-OF-THOUGHT:")
print(f"   {r3['response'][:300]}")

1. ZERO-SHOT:
   The capital of Australia is Canberra.

2. FEW-SHOT:
   Canberra

3. CHAIN-OF-THOUGHT:
   I'm happy to help you with that!

The capital of Australia is actually Canberra, not Sydney or Melbourne (which are both major cities in Australia). Here's why:

1. Australia is a federal country, meaning it has a system of government where power is divided between the central government and smaller


## 7. Advanced: Output Guardrails & Constraints

In production, you need the model to stay WITHIN bounds. Here are techniques for controlling output:

In [14]:
# Technique: Constraining the response with strong instructions
constrained = ollama.chat(model=MODEL, messages=[
    {'role': 'system', 'content': """You are a land use classifier. 
You MUST respond with EXACTLY one word from this list: Urban, Agricultural, Forest, Water, Barren.
Do not explain. Do not add punctuation. Just the single classification word."""},
    {'role': 'user', 'content': 'A wide expanse of golden wheat fields stretching to the horizon with a few tractors visible.'}
])

classification = constrained['message']['content'].strip()
print(f"Classification: '{classification}'")
print(f"Valid? {classification in ['Urban', 'Agricultural', 'Forest', 'Water', 'Barren']}")

Classification: 'Agricultural'
Valid? True


## Quick Reference: Which Technique When?

| Task Type | Best Technique | Why |
| :--- | :--- | :--- |
| Simple, well-known task | **Zero-shot** | Model already knows how |
| Specific format needed | **Few-shot** | Examples teach the format |
| Math, logic, reasoning | **Chain-of-Thought** | Scratchpad prevents errors |
| Consistent behavior | **System prompt** | Sets global rules |
| Data extraction | **Few-shot + JSON** | Examples + forced format |
| Complex, multi-step | **CoT + System** | Combine techniques! |

## Challenge! 🚀

1. Create a system prompt that makes the model act as a **Remote Sensing Expert**. Ask it to explain the difference between optical and SAR imagery.
2. Use few-shot prompting to build a **tweet classifier** that labels tweets as: `Disaster`, `Weather`, `Traffic`, or `Other`.
3. Try combining techniques: Use a system prompt + few-shot examples + chain-of-thought on a geospatial reasoning problem.

---

**Next notebook:** [HuggingFace Local](./huggingface_local.ipynb) — Run models without Ollama using the industry-standard `transformers` library.